<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 13 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Partition Maintenance and Recovery</div>
</div>

Orders keep arriving, so old months have to go, and a mistake has to be undone. On a table of its own with three monthly partitions, you record the evidence that a maintenance change needs, clear January with `TRUNCATE` and see which IDs change, then bring January back from the recycle bin by the ID of the partition that the truncation removed. You compare a `DROP PARTITION` that the recycle bin can undo, and check a retention plan in SQL before it removes anything. The optional sections 6 to 8 back a table up to a local object store, drop it for good, and restore it in another database. Plan about 60 minutes; the optional sections add about 15.

**Module 13: Storage and Lifecycle Management** · [Lesson](course.md) · [Quiz](quiz13_storage_lifecycle.ipynb) · [Course README](../../README.md)

## Before you begin

This Lab needs only the course sandbox: no earlier Lab's tables. It rebuilds `ops_orders_l3` and `ops_orders_drop_l3`, and in the practice `retention_plan_l3`. The partitions that it truncates and drops stay in Doris's recycle bin for a day (FE configuration `catalog_trash_expire_second`, 86400 seconds), then Doris deletes them.

The optional sections 6 to 8 run only when you switch them on. They start the Compose project `doris-warehousing-backup`, an object store with the container `doris-warehousing-backup-silo-1` and the volume `backup-objects`, and they create the repository `<your database>_backup_l3` and the database `<your database>_restore_l3`. These stay after the Lab, so that you can look at the snapshot; section 8 says how to remove them.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import pymysql

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import card, show_records, show_sql

lab = connect_sandbox(module=13)
PARTITION_COLUMNS = ["PartitionId", "PartitionName", "VisibleVersion", "State", "Range", "Buckets", "ReplicationNum", "RowCount"]

## 1. Create a monthly table of its own

Partitions define lifecycle boundaries. HASH distribution determines the tablets within each partition. This sandbox uses one bucket and one replica to illustrate the layout; these values are not a production sizing recommendation.

The setup drops the previous Lab table so the Notebook can be rerun. Read the table's definition (its DDL, Data Definition Language) and its boundaries before you run it.

In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_l3")
lab.execute("""
CREATE TABLE ops_orders_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_l3", ["order_date", "order_id", "status", "amount"], [("2025-01-15", 130001, "PAID", 100), ("2025-02-15", 130002, "PAID", 200), ("2025-03-15", 130003, "SHIPPED", 300)])
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="The lifecycle table")
expect(lab.query("""
SELECT CAST(order_date AS STRING), order_id, status, amount
FROM ops_orders_l3
ORDER BY order_date, order_id
"""), [("2025-01-15", 130001, "PAID", "100.00"), ("2025-02-15", 130002, "PAID", "200.00"), ("2025-03-15", 130003, "SHIPPED", "300.00")],
       title="One order in each of three months")

**Expected result**

Three orders: 130001 on 2025-01-15 (PAID, 100.00), 130002 on 2025-02-15 (PAID, 200.00), and 130003 on 2025-03-15 (SHIPPED, 300.00), and a green check. Each lies in its own partition: `p202501`, `p202502`, and `p202503`.

## 2. Record the maintenance evidence

Before a change, record what it may touch, so that the same checks afterward show what it did:

- `SHOW PARTITIONS`: each partition's name, range, ID, and visible version.
- `SHOW TABLETS`: each tablet's ID, backend, and version. Each row is one replica; the output has no partition column, so add `PARTITION (...)` to see one partition's tablets.
- `SHOW CREATE TABLE`: the key model, the partitions, the bucketing, and the replicas.

`RowCount` comes from the BE's periodic report and can be -1 or 0 for a minute after a load; `VisibleVersion` changes at once. For an exact count now, query the partition: `SELECT COUNT(*) FROM ops_orders_l3 PARTITION (p202501)`. One bucket and one replica suit this sandbox; they are not a size for production.

In [ ]:
before_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions before maintenance",
                            columns=PARTITION_COLUMNS)
lab.sql("SHOW TABLETS FROM ops_orders_l3", title="The tablets: one row per replica",
        columns=["TabletId", "BackendId", "Version", "RowCount", "State"])
lab.sql("SHOW CREATE TABLE ops_orders_l3", title="The table's definition")
expect([name for name in before_partitions["PartitionName"]], ["p202501", "p202502", "p202503"], title="Three monthly partitions")
expect([lab.query(f"SELECT COUNT(*) FROM ops_orders_l3 PARTITION ({name})")[0][0] for name in ("p202501", "p202502", "p202503")],
       [1, 1, 1], title="One order in each partition, counted now")

**Expected result**

Three partitions, `p202501`, `p202502`, and `p202503`, each `NORMAL` at `VisibleVersion` 2 with 1 bucket and 1 replica, and `RowCount` -1 or 0 if the BE has not reported yet, otherwise 1. Three tablets, one per partition, on the one backend, at version 2. The definition: `DUPLICATE KEY(order_date, order_id)`, three `RANGE` partitions, `DISTRIBUTED BY HASH(order_id) BUCKETS 1`. Two green checks: three partitions, and one order in each, counted now.

Note January's `PartitionId`: the next step changes it, and step 4 needs the old one.

## 3. Clear January only

`TRUNCATE TABLE ... PARTITION (p202501)` empties one partition and keeps its definition, so the month can be loaded again. Doris replaces the partition with a new, empty one of the same name and range, and moves the old one, with its data, to the recycle bin. It does not delete rows one by one and does not wait for compaction: the old rows are gone from queries at once.

Predict: which partition gets a new `PartitionId`, and which keep theirs?

In [ ]:
lab.execute("TRUNCATE TABLE ops_orders_l3 PARTITION (p202501)")
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="The rows after clearing January")
after_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions after maintenance",
                           columns=PARTITION_COLUMNS)
before_ids = before_partitions.set_index("PartitionName")["PartitionId"].to_dict()
after_ids = after_partitions.set_index("PartitionName")["PartitionId"].to_dict()
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_l3"), [(2, "500.00")], title="February and March remain: 2 orders, 500.00")
expect((before_ids["p202501"] != after_ids["p202501"], [before_ids[n] == after_ids[n] for n in ("p202502", "p202503")]),
       (True, [True, True]), title="January has a new PartitionId; February and March keep theirs")
expect([lab.query(f"SELECT COUNT(*) FROM ops_orders_l3 PARTITION ({name})")[0][0] for name in ("p202501", "p202502", "p202503")],
       [0, 1, 1], title="January is empty; the other months are unchanged")

**Expected result**

The rows of February and March, then the partitions: `p202501` with a new `PartitionId` and `VisibleVersion` 1, `p202502` and `p202503` as before. Three green checks: 2 orders and 500.00 remain, only January has a new ID, and January is empty while the other months keep their order.

The query result changed at once; the old partition's files did not go anywhere yet: they are in the recycle bin, which the next step uses.

## 4. Recover the truncated January by its partition ID

The truncation was a mistake: January must come back. The recycle bin still has the old partition, under the ID that step 2 recorded. Three things stand in the way, and the cell meets them in order:

1. `SHOW CATALOG RECYCLE BIN WHERE NAME = 'p202501'` lists every dropped partition of that name, of every table and every run of this Lab; the cell keeps the entry with the recorded ID.
2. `RECOVER PARTITION p202501 FROM ops_orders_l3` fails while the empty replacement of the same name is in the table. Before you remove the replacement, check that it is still empty: rows written to it after the truncation would be lost.
3. Dropping the replacement puts it in the recycle bin too. A `RECOVER` by name would now bring back the newest `p202501` there, the empty one, so the cell recovers by the ID: `RECOVER PARTITION p202501 <id> FROM ops_orders_l3`.

In [ ]:
old_id = before_ids["p202501"]
with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
    cursor.execute("SHOW CATALOG RECYCLE BIN WHERE NAME = 'p202501'")
    entries = cursor.fetchall()
show_records("The recycle bin's entry of the truncated January", [entry for entry in entries if str(entry["PartitionId"]) == str(old_id)])
try:
    lab.execute("RECOVER PARTITION p202501 FROM ops_orders_l3")
    blocked = None
except pymysql.err.OperationalError as error:
    blocked = str(error.args[-1]).split("detailMessage = ", 1)[-1]
expect(blocked is not None, True, title="RECOVER by name is refused while the replacement exists")
expect(lab.query("SELECT COUNT(*) FROM ops_orders_l3 PARTITION (p202501)"), [(0,)], title="The replacement is still empty")
recover_sql = ["ALTER TABLE ops_orders_l3 DROP PARTITION p202501", f"RECOVER PARTITION p202501 {old_id} FROM ops_orders_l3"]
show_sql("Remove the empty replacement, then recover January by its ID", ";\n".join(recover_sql) + ";")
for statement in recover_sql:
    lab.execute(statement)
recovered = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions after the recovery",
                    columns=PARTITION_COLUMNS)
show_records("What Doris answered the RECOVER by name", [{"statement": "RECOVER PARTITION p202501 FROM ops_orders_l3", "answer": blocked}])
expect((recovered.set_index("PartitionName")["PartitionId"].to_dict()["p202501"], lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_l3")),
       (old_id, [(3, "600.00")]), title="January is back with its original ID and order: 3 orders, 600.00")

**Expected result**

The recycle bin's entry: type `Partition`, name `p202501`, the IDs of your database and table, the recorded `PartitionId`, and the time of the truncation. A green check: the `RECOVER` by name is refused, with `partition[p202501] already exist in table[ops_orders_l3]`; a green check: the replacement is still empty. The two statements, then the partitions: `p202501` again with its original `PartitionId` and `VisibleVersion` 2. A green check: January is back, and the table has 3 orders and 600.00.

Measured on Doris 4.1.3: after the replacement was dropped, a `RECOVER` by name would have recovered the replacement, the newest partition of that name, and January would have stayed empty. The recycle bin is a time-limited way back, for one day here; the durable one is a backup or an archive that you have read back.

## 5. DROP a partition, and RECOVER it

`TRUNCATE` left an empty partition of the same name, for a reload. `DROP PARTITION` removes the partition and its definition, which is what an expired month needs after its archive has been checked.

Use a second isolated table for the comparison. Predict the results first: after dropping January, SHOW PARTITIONS lists only February. Without FORCE, recovery within recycle-bin retention restores January and its original row.

This temporary recovery path does not replace a durable backup.

In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_drop_l3")
lab.execute("""
CREATE TABLE ops_orders_drop_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_drop_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 130101, 100), ("2025-02-15", 130102, 200)])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(2, "300.00")], title="The second table has two months: 2 orders, 300.00")
lab.execute("ALTER TABLE ops_orders_drop_l3 DROP PARTITION p202501")
dropped_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_drop_l3 ORDER BY PartitionName", title="The partitions after the DROP", columns=PARTITION_COLUMNS);
expect("p202501" in set(dropped_partitions["PartitionName"]), False, title="The DROP removed January's definition")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(1, "200.00")], title="Only February's 200.00 remains")
lab.execute("RECOVER PARTITION p202501 FROM ops_orders_drop_l3")
recovered_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_drop_l3 ORDER BY PartitionName", title="The partitions after the RECOVER", columns=PARTITION_COLUMNS);
expect("p202501" in set(recovered_partitions["PartitionName"]), True, title="The RECOVER brought January's definition back")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(2, "300.00")], title="And its order: 2 orders, 300.00")

**Expected result**

Five green checks: the second table has two months, 2 orders and 300.00; after the `DROP` its partitions list only `p202502`, and only February's 200.00 remains; after the `RECOVER`, `p202501` is listed again and the table has 2 orders and 300.00. The two tables of partitions show February alone, then both months.

This `DROP` has no `FORCE`, so the partition went to the recycle bin and could come back: here a `RECOVER` by name is enough, because no other `p202501` of this table is there. With `FORCE`, or after the retention, only a backup or an archive could bring it back.

## 6. Optional: back up a table to a repository

**This section and sections 7 and 8 are optional, and they run only when you switch them on**: set `RUN_BACKUP_DRILL = True` in the next cell, then run it and the cell after it. Skipping them does not affect Quiz 13 or Module 14.

The earlier steps used Doris's recycle bin. Now make a **real full repository backup**, drop an isolated source table forcibly, and restore its snapshot into a separate course database. Forced deletion intentionally removes this table's recycle-bin fallback; the repository snapshot is the recovery source.

This section explicitly starts a course-owned Silo service, the S3-compatible object store also used in Lab 4: at most 512 MiB RAM, loopback port 51910 (`DW_BACKUP_S3_PORT`; start Jupyter with it set to use another), and a persistent `backup-objects` volume. The helper checks the port before it starts anything. It uses public local-only teaching credentials, never production credentials. Both the FE and BE must reach its container endpoint. The table to back up is an ordinary table of this integrated storage and compute sandbox, without a Storage Policy or an asynchronous materialized view.

**Boundary:** Silo and Doris still run on one host. This validates the repository protocol and data reconciliation, not host-loss disaster recovery. Production backups need independent failure domains, credential management, retention and tested recovery objectives.

In [ ]:
# Optional: set to True, then run this cell and the next one, to back up and restore a table.
RUN_BACKUP_DRILL = False

if RUN_BACKUP_DRILL:
    import secrets
    from dw_course.backup import prepare_backup_storage, BUCKET, ACCESS_KEY, SECRET_KEY
    from dw_course.streaming import wait_for

    backup_client = prepare_backup_storage(lab, start=True)
    repository = lab.database + "_backup_l3"
    restore_database = lab.database + "_restore_l3"
    backup_label = "course_snapshot_" + secrets.token_hex(6)
    existing_repositories = {row[1] for row in lab.query("SHOW REPOSITORIES")}
    if repository in existing_repositories:
        lab.execute(f"DROP REPOSITORY {repository}")
    lab.execute(f"""CREATE REPOSITORY {repository} WITH S3
    ON LOCATION "s3://{BUCKET}/{lab.database}/repository"
    PROPERTIES ("s3.endpoint"="http://course-backup-silo:9000", "s3.region"="us-east-1",
                "s3.access_key"="{ACCESS_KEY}", "s3.secret_key"="{SECRET_KEY}", "use_path_style"="true")""")
    lab.execute("DROP TABLE IF EXISTS backup_source_l3")
    lab.execute("""CREATE TABLE backup_source_l3 (
        order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
    ) DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
    PROPERTIES ("replication_num"="1")""")
    backup_rows = [(7301,"100.00"),(7302,"200.00"),(7303,"300.00")]
    lab.insert("backup_source_l3", ["order_id","amount"], backup_rows)
    expect(lab.query("SELECT * FROM backup_source_l3 ORDER BY order_id"), backup_rows, title="The source table holds the three rows to back up")
    lab.execute(f'BACKUP SNAPSHOT {lab.database}.{backup_label} TO {repository} ON (backup_source_l3) PROPERTIES("type"="full")')

    def repository_job(kind, database):
        with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
            cursor.execute(f"SHOW {kind} FROM {database}")
            name_column = "SnapshotName" if kind=="BACKUP" else "Label"
            jobs = [row for row in cursor.fetchall() if row[name_column] == backup_label]
        if not jobs:
            return None
        current = jobs[-1]
        if current["State"] == "CANCELLED":
            raise RuntimeError(f"{kind} failed: {current}")
        return current

    backup_job = wait_for(lambda: repository_job("BACKUP",lab.database),
                          lambda row: row is not None and row["State"]=="FINISHED",
                          description="Upload the full snapshot to the repository", timeout=180)
    lab.sql(f"SHOW BACKUP FROM {lab.database}", title="The backup jobs of your database; this run's is the last");
else:
    card("Set RUN_BACKUP_DRILL = True in this cell and run it again to start the object store and back up a table.",
         "skip", "The optional backup drill was not run")

**Expected result**

With `RUN_BACKUP_DRILL = False`: a gray card, the optional drill was not run.

With `True`: the three steps of the object store's start, all green; a green check of the three source rows, 7301 to 7303; a progress card while the snapshot is uploaded; then the backup jobs of your database: the last one, this run's, is `FINISHED` with the snapshot's name `course_snapshot_<random>`; earlier runs of the drill are listed above it.

## 7. Optional: restore into another database and reconcile every row

A successful upload is not a tested restore. Obtain the repository's actual snapshot timestamp rather than inventing one, then restore into a database whose name remains under the course's allowed prefix. `ON (... AS ...)` names the single restored table; other tables and the main course database are not reset.

The source is deliberately force-dropped only after the backup reaches `FINISHED`. Check restore completion, ordered rows, count and amount. A snapshot recovers its recorded cutoff, not writes that arrived later; a lower recovery-point objective needs another data-replay strategy.

In [ ]:
if RUN_BACKUP_DRILL:
    snapshot = lab.sql(f"SHOW SNAPSHOT ON {repository} WHERE SNAPSHOT='{backup_label}'", title="Repository snapshot and real backup timestamp")
    expect(snapshot.iloc[0]["Status"], "OK")
    backup_timestamp = snapshot.iloc[0]["Timestamp"]
    lab.execute("DROP TABLE backup_source_l3 FORCE")
    lab.execute(f"CREATE DATABASE IF NOT EXISTS {restore_database}")
    lab.execute(f"DROP TABLE IF EXISTS {restore_database}.restored_orders_l3")
    lab.execute(f"""RESTORE SNAPSHOT {restore_database}.{backup_label} FROM {repository}
    ON (backup_source_l3 AS restored_orders_l3)
    PROPERTIES ("backup_timestamp"="{backup_timestamp}", "replication_num"="1")""")
    restore_job = wait_for(lambda: repository_job("RESTORE",restore_database),
                           lambda row: row is not None and row["State"]=="FINISHED",
                           description="Restore the repository snapshot into an isolated target", timeout=180)
    expect(lab.query(f"SELECT * FROM {restore_database}.restored_orders_l3 ORDER BY order_id"), backup_rows, title="Restored rows match the independent backup-cutoff ledger")
    expect(lab.query(f"SELECT COUNT(*),SUM(amount) FROM {restore_database}.restored_orders_l3"), [(3,"600.00")], title="The restored table has 3 orders and 600.00")
    show_records("Recovery source comparison", [
        {"operation":"Earlier RECOVER drill","source":"recycle bin","boundary":"retention and original metadata must still exist"},
        {"operation":"This RESTORE drill","source":"full repository snapshot","boundary":"snapshot cutoff; no post-snapshot writes"},
    ])
    lab.sql(f"SHOW RESTORE FROM {restore_database}", title="The restore jobs; this run's is the last");
else:
    card("Run the previous cell with RUN_BACKUP_DRILL = True first.", "skip", "The optional restore was not run")

**Expected result**

With the drill off: a gray card.

With it on: the repository's snapshot with `Status` OK and its real backup timestamp; a progress card while the snapshot is restored; two green checks, the restored table has the three rows 7301 to 7303, 3 orders and 600.00; the comparison of the two ways back, the recycle bin and the repository; and the restore jobs, the last one this run's, `FINISHED`.

The source table was dropped with `FORCE`, so the recycle bin could not help: the snapshot did. It holds what the table had when the backup ran, and nothing written later.

## 8. Optional: keep the snapshot, and own its cleanup

The repository, the snapshot objects in Silo and the isolated restored table remain for review. Rerunning creates a new snapshot label and rebuilds only the named source/target tables. It never deletes the object-store volume. Run `docker compose --project-name doris-warehousing-backup --file environments/backup/compose.yml stop` from the course root to release service RAM while retaining objects.

Dropping a Repository removes Doris's registration, not the remote snapshot data. Delete a backup only under an explicit retention decision; stopping the service is not deletion. This drill does not back up the whole cluster, the users' grants, or external lake data. To remove everything that it created, run `DROP REPOSITORY <your database>_backup_l3` and `DROP DATABASE <your database>_restore_l3` in Doris, then, from the course directory, `docker compose --project-name doris-warehousing-backup --file environments/backup/compose.yml down --volumes`; the course README's Clean up section lists the order.

## 9. Practice: check a retention plan before it removes anything

The order table keeps the current calendar month and the 12 months before it. Maintenance runs on 2026-02-15, so February 2025 to February 2026 stay and the months before go. Create `retention_plan_l3` with one partition per month from November 2024 to February 2026 and one order in each; compute the first month to keep in SQL from the maintenance date; list the partitions that end on or before it; drop them without `FORCE`; and check that 13 months and 13 orders remain, from February 2025 to February 2026.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
months = [f"{2024 + (10 + i) // 12}-{(10 + i) % 12 + 1:02d}" for i in range(16)]  # 2024-11 ... 2026-02
bounds = months[1:] + ["2026-03"]
partitions = ",\n    ".join(f'PARTITION p{m.replace("-", "")} VALUES [("{m}-01"), ("{b}-01"))' for m, b in zip(months, bounds))
plan_ddl = f"""CREATE TABLE retention_plan_l3 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    {partitions}
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES ("replication_num"="1")"""
lab.execute("DROP TABLE IF EXISTS retention_plan_l3")
show_sql("A table of 16 monthly partitions", plan_ddl)
lab.execute(plan_ddl)
lab.insert("retention_plan_l3", ["order_date", "order_id", "amount"], [(f"{m}-15", 150000 + i, 10) for i, m in enumerate(months)])
keep_from = lab.query("SELECT CAST(DATE_TRUNC(DATE_SUB(DATE '2026-02-15', INTERVAL 12 MONTH), 'month') AS STRING)")[0][0]
expired = [f"p{m.replace('-', '')}" for m, b in zip(months, bounds) if f"{b}-01" <= keep_from]
show_records("The plan", [{"maintenance_date": "2026-02-15", "first_month_kept": keep_from, "partitions_to_drop": ", ".join(expired)}])
for name in expired:
    lab.execute(f"ALTER TABLE retention_plan_l3 DROP PARTITION {name}")
expect((keep_from, expired), ("2025-02-01", ["p202411", "p202412", "p202501"]), title="The plan drops November 2024 to January 2025")
expect(lab.query("SELECT COUNT(*), CAST(MIN(order_date) AS STRING), CAST(MAX(order_date) AS STRING) FROM retention_plan_l3"),
       [(13, "2025-02-15", "2026-02-15")], title="13 months remain, February 2025 to February 2026")

**Expected result**

The definition of 16 partitions, `p202411` to `p202602`, then the plan: maintenance on 2026-02-15, the first month kept 2025-02-01, the partitions to drop `p202411, p202412, p202501`. Two green checks: the plan drops those three, and 13 orders remain, from 2025-02-15 to 2026-02-15.

The rule is in calendar months: a rolling window of 365 days from the maintenance date would end on another day and could not be applied by dropping whole months. Before a real run, add the checks of step 2 and a read-back of each month's archive; without `FORCE`, the dropped months are in the recycle bin for a day.

## You are done when

- you can say what `SHOW PARTITIONS`, `SHOW TABLETS`, `SHOW CREATE TABLE`, and a partition's `COUNT(*)` each show, and which of their numbers lag;
- `TRUNCATE` emptied January alone and gave it a new `PartitionId`, and you can say why the query result changed before any file was deleted;
- you recovered the truncated January by its ID, and can say why a `RECOVER` by name would have failed, then recovered the wrong partition;
- you can choose between `TRUNCATE` and `DROP PARTITION` for a month to reload and a month that expired, and say what the recycle bin does and does not replace;
- your retention plan dropped November 2024 to January 2025 and kept 13 months.

## Next

Open [Quiz 13: Storage and Lifecycle Management](quiz13_storage_lifecycle.ipynb), then continue to [Module 14: Resource Isolation and Query Protection](../module14-resource-isolation/course.md).